In [1]:
import requests
print("ok")

ok


In [2]:
import requests, json

CATEGORIES = ["hardware_fault", "service_outage", "billing", "provisioning", "performance_degradation"]

FEW_SHOT = """
<example>
<ticket>Customer's router keeps rebooting every few hours, replaced twice already.</ticket>
<output>{"category": "hardware_fault", "urgency": "medium", "confidence": 0.9}</output>
</example>
<example>
<ticket>Entire apartment block has no internet since 3am, confirmed by 6 separate customers.</ticket>
<output>{"category": "service_outage", "urgency": "high", "confidence": 0.95}</output>
</example>
"""

def classify_ticket(ticket_text: str) -> dict:
    prompt = f"""You classify telecom support tickets. Categories: {CATEGORIES}.

{FEW_SHOT}

Now classify this ticket. Respond with ONLY a JSON object, no markdown fences, no other text.

<ticket>{ticket_text}</ticket>
<output>"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0},
        },
    )
    raw = response.json()["response"].strip()
    print("RAW OUTPUT:", repr(raw))
    return json.loads(raw)

tickets = [
    "My bill jumped from $60 to $140 with no explanation this month.",
    "Download speeds dropped to 2Mbps during evenings only, was getting 100Mbps.",
    "New fiber install scheduled, technician never showed up.",
]
for t in tickets:
    print(t, "->", classify_ticket(t))
    print()

RAW OUTPUT: '{"category": "billing", "urgency": "low", "confidence": 0.8}'
My bill jumped from $60 to $140 with no explanation this month. -> {'category': 'billing', 'urgency': 'low', 'confidence': 0.8}

RAW OUTPUT: '{"category": "performance_degradation", "urgency": "medium", "confidence": 0.8}'
Download speeds dropped to 2Mbps during evenings only, was getting 100Mbps. -> {'category': 'performance_degradation', 'urgency': 'medium', 'confidence': 0.8}

RAW OUTPUT: '{"category": "service_outage", "urgency": "high", "confidence": 0.9}'
New fiber install scheduled, technician never showed up. -> {'category': 'service_outage', 'urgency': 'high', 'confidence': 0.9}



In [3]:
edge_cases = [
    "Technician never showed up for my install.",
    "My fiber install technician came but said the wrong equipment was ordered.",
    "Outage last night, and now my bill shows a charge for that day anyway.",
]

for t in edge_cases:
    print(t, "->", classify_ticket(t))
    print()

RAW OUTPUT: '{"category": "provisioning", "urgency": "low", "confidence": 0.8}'
Technician never showed up for my install. -> {'category': 'provisioning', 'urgency': 'low', 'confidence': 0.8}

RAW OUTPUT: '{"category": "provisioning", "urgency": "medium", "confidence": 0.8}'
My fiber install technician came but said the wrong equipment was ordered. -> {'category': 'provisioning', 'urgency': 'medium', 'confidence': 0.8}

RAW OUTPUT: '{"category": "billing", "urgency": "low", "confidence": 0.8}'
Outage last night, and now my bill shows a charge for that day anyway. -> {'category': 'billing', 'urgency': 'low', 'confidence': 0.8}



In [4]:
# Run the exact same ticket 3 times in a row — is it truly deterministic?
for i in range(3):
    print(classify_ticket("Technician never showed up for my install."))

RAW OUTPUT: '{"category": "provisioning", "urgency": "low", "confidence": 0.8}'
{'category': 'provisioning', 'urgency': 'low', 'confidence': 0.8}
RAW OUTPUT: '{"category": "provisioning", "urgency": "low", "confidence": 0.8}'
{'category': 'provisioning', 'urgency': 'low', 'confidence': 0.8}
RAW OUTPUT: '{"category": "provisioning", "urgency": "low", "confidence": 0.8}'
{'category': 'provisioning', 'urgency': 'low', 'confidence': 0.8}
